#### Iris Data Aalysis

##### Swapnanil Bala, NUID: 002031137, Khoury College of Computer Sciences MSDS

In [1]:
# This is to check what libraries I have currently installed
%pip list

Package                   Version      Editable project location
------------------------- ------------ ------------------------------
accelerate                1.13.0
aiofiles                  24.1.0
aiohappyeyeballs          2.6.1
aiohttp                   3.13.3
aiohttp_socks             0.11.0
aiosignal                 1.4.0
altair                    6.0.0
altgraph                  0.17.5
annotated-doc             0.0.4
annotated-types           0.7.0
anyio                     4.12.1
argon2-cffi               25.1.0
argon2-cffi-bindings      25.1.0
arrow                     1.4.0
asttokens                 3.0.1
async-lru                 2.2.0
attrs                     25.4.0
babel                     2.18.0
bcrypt                    5.0.0
beautifulsoup4            4.14.3
bleach                    6.3.0
blinker                   1.9.0
cachetools                7.0.3
certifi                   2026.2.25
cffi                      2.0.0
charset-normalizer        3.4.5
click             

## Metrics

General Rules: "positive": **TP** correctly caught, **FP** false alarm, **FN** missed, **TN** correctly rejected.

| Metric | Formula | Question it answers |
|---|---|---|
| **Accuracy** | (TP+TN) / total | What fraction of all predictions were right? |
| **Precision** | TP / (TP+FP) | Of what I flagged, how much was real? |
| **Recall** | TP / (TP+FN) | Of what was real, how much did I catch? |
| **F1** | 2·P·R / (P+R) | Harmonic mean — high only if *both* are high |

- Accuracy misleads on imbalanced data; a 99%-negative dataset scores 99% by predicting nothing.
- Precision vs recall trade off: catch more positives, admit more junk.
- F1 is harmonic, not arithmetic, so one collapsed score drags it down (P=1.0, R=0 → F1=0, not 0.5).
- Multi-class: computed per class, then averaged — **macro** (equal weight) or **weighted** (by support). Always say which.
- **Support** = true instances of that class; small support = noisy score.

In [14]:
# For our task we will require a few packages, starting from the next line we have the import codes.

import time # This is to check the training time
 
import numpy as np # The classic Numpy module
from sklearn.datasets import load_iris # we are importing the iris dataset
from sklearn.model_selection import train_test_split # This library is the one which we will use to split the data into training and testing portion
from sklearn.preprocessing import StandardScaler # Rescales each feature to mean 0, std 1 so no single feature dominates KNN's distance metric
from sklearn.neighbors import KNeighborsClassifier # A data mining technique for classifying unique clusters of data
from sklearn.tree import DecisionTreeClassifier  # A data mining technique for classifying unique clusters of data
from sklearn.naive_bayes import GaussianNB # A data mining technique for classifying unique clusters of data
from sklearn.linear_model import LogisticRegression # A data mining technique for classifying unique clusters of data
from sklearn.ensemble import RandomForestClassifier # A data mining technique for classifying unique clusters of data
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score # our metrics on which the model wil be judged

In [3]:
# 1. Loading the iris daatset itself
iris = load_iris() # loading the iris dataset, and assigning a pointer (variable since this is python) iris to it

X = iris.data # This is the data
y = iris.target # The answers

In [4]:
# 2. Splitting the dataset prior to touching the data any other way.
#    stratify=y keeps all three species balanced in both halves.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state= 222
)

In [5]:
# 3. Scaling: fitting on TRAIN only, then transform both.
#    Fitting on the full X leaks test information into training.
scaler = StandardScaler().fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)

In [6]:
# 4. Model — we have chosen the K-Neighbors one and we can set the neighbor to our will
model_1 = KNeighborsClassifier(n_neighbors=5)

In [7]:
# 5. Training time

N_RUNS = 50 # N_RUNS is essentially how many iterations of training we will do

train_times = [] # This essentially store the time taken for our trainings
for _ in range(N_RUNS):
    t0 = time.perf_counter() # timer starts here
    model_1.fit(X_train, y_train)
    train_times.append(time.perf_counter() - t0) # in the end we substract the start time from it

In [8]:
# 6. Testing time 
test_times = [] # This essentially store the time taken for our testings
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    y_pred = model_1.predict(X_test)
    test_times.append(time.perf_counter() - t0)

In [10]:
# 7. (c) Accuracy for the KNeighborsClassifier
acc = accuracy_score(y_test, y_pred)
 
print(f"Model:           {model_1.__class__.__name__}")
print(f"Train / Test:    {len(X_train)} / {len(X_test)} samples")
print(f"Training time:   {np.mean(train_times)*1000:8.4f} ms  "
      f"(sd {np.std(train_times)*1000:.4f}, n={N_RUNS})")
print(f"Testing time:    {np.mean(test_times)*1000:8.4f} ms  "
      f"(sd {np.std(test_times)*1000:.4f}, n={N_RUNS})")
print(f"Test accuracy:   {acc:.4f}  ({acc*100:.2f}%)")
print()
print(classification_report(y_test, y_pred, target_names=iris.target_names))


Model:           KNeighborsClassifier
Train / Test:    120 / 30 samples
Training time:     0.2399 ms  (sd 0.0964, n=50)
Testing time:      0.6059 ms  (sd 0.1794, n=50)
Test accuracy:   0.9000  (90.00%)

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.82      0.90      0.86        10
   virginica       0.89      0.80      0.84        10

    accuracy                           0.90        30
   macro avg       0.90      0.90      0.90        30
weighted avg       0.90      0.90      0.90        30



In [11]:
# Trying to use the Decision Tree Classifier now, keeping the rest mostly the same
model_2 = DecisionTreeClassifier(random_state=42)

In [12]:
# 8. 
N_RUNS = 50 # N_RUNS is essentially how many iterations of training we will do

train_times = [] # This essentially store the time taken for our trainings
for _ in range(N_RUNS):
    t0 = time.perf_counter() # timer starts here
    model_2.fit(X_train, y_train)
    train_times.append(time.perf_counter() - t0) # in the end we substract the start time from it


test_times = [] # This essentially store the time taken for our testings
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    y_pred = model_2.predict(X_test)
    test_times.append(time.perf_counter() - t0)

In [13]:
# 9. Accuracy for the Decision_Tree

acc = accuracy_score(y_test, y_pred)
 
print(f"Model:           {model_2.__class__.__name__}")
print(f"Train / Test:    {len(X_train)} / {len(X_test)} samples")
print(f"Training time:   {np.mean(train_times)*1000:8.4f} ms  "
      f"(sd {np.std(train_times)*1000:.4f}, n={N_RUNS})")
print(f"Testing time:    {np.mean(test_times)*1000:8.4f} ms  "
      f"(sd {np.std(test_times)*1000:.4f}, n={N_RUNS})")
print(f"Test accuracy:   {acc:.4f}  ({acc*100:.2f}%)")
print()
print(classification_report(y_test, y_pred, target_names=iris.target_names))

Model:           DecisionTreeClassifier
Train / Test:    120 / 30 samples
Training time:     0.4349 ms  (sd 0.1958, n=50)
Testing time:      0.0376 ms  (sd 0.0087, n=50)
Test accuracy:   0.9000  (90.00%)

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.82      0.90      0.86        10
   virginica       0.89      0.80      0.84        10

    accuracy                           0.90        30
   macro avg       0.90      0.90      0.90        30
weighted avg       0.90      0.90      0.90        30



In [15]:
# GaussianNB
model_3 =  GaussianNB()

train_times = [] # This essentially store the time taken for our trainings
for _ in range(N_RUNS):
    t0 = time.perf_counter() # timer starts here
    model_3.fit(X_train, y_train)
    train_times.append(time.perf_counter() - t0) # in the end we substract the start time from it


test_times = [] # This essentially store the time taken for our testings
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    y_pred = model_3.predict(X_test)
    test_times.append(time.perf_counter() - t0)

acc = accuracy_score(y_test, y_pred)
 
print(f"Model:           {model_3.__class__.__name__}")
print(f"Train / Test:    {len(X_train)} / {len(X_test)} samples")
print(f"Training time:   {np.mean(train_times)*1000:8.4f} ms  "
      f"(sd {np.std(train_times)*1000:.4f}, n={N_RUNS})")
print(f"Testing time:    {np.mean(test_times)*1000:8.4f} ms  "
      f"(sd {np.std(test_times)*1000:.4f}, n={N_RUNS})")
print(f"Test accuracy:   {acc:.4f}  ({acc*100:.2f}%)")
print()
print(classification_report(y_test, y_pred, target_names=iris.target_names))

Model:           GaussianNB
Train / Test:    120 / 30 samples
Training time:     0.4268 ms  (sd 0.1723, n=50)
Testing time:      0.0675 ms  (sd 0.0095, n=50)
Test accuracy:   0.9667  (96.67%)

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.91      1.00      0.95        10
   virginica       1.00      0.90      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30



In [16]:
# Logistic Regression 

model_4 = LogisticRegression(max_iter=3000, random_state=42)

train_times = [] # This essentially store the time taken for our trainings
for _ in range(N_RUNS):
    t0 = time.perf_counter() # timer starts here
    model_4.fit(X_train, y_train)
    train_times.append(time.perf_counter() - t0) # in the end we substract the start time from it


test_times = [] # This essentially store the time taken for our testings
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    y_pred = model_4.predict(X_test)
    test_times.append(time.perf_counter() - t0)

acc = accuracy_score(y_test, y_pred)
 
print(f"Model:           {model_4.__class__.__name__}")
print(f"Train / Test:    {len(X_train)} / {len(X_test)} samples")
print(f"Training time:   {np.mean(train_times)*1000:8.4f} ms  "
      f"(sd {np.std(train_times)*1000:.4f}, n={N_RUNS})")
print(f"Testing time:    {np.mean(test_times)*1000:8.4f} ms  "
      f"(sd {np.std(test_times)*1000:.4f}, n={N_RUNS})")
print(f"Test accuracy:   {acc:.4f}  ({acc*100:.2f}%)")
print()
print(classification_report(y_test, y_pred, target_names=iris.target_names))

Model:           LogisticRegression
Train / Test:    120 / 30 samples
Training time:     2.2297 ms  (sd 1.0452, n=50)
Testing time:      0.0370 ms  (sd 0.0156, n=50)
Test accuracy:   0.9667  (96.67%)

              precision    recall  f1-score   support

      setosa       1.00      1.00      1.00        10
  versicolor       0.91      1.00      0.95        10
   virginica       1.00      0.90      0.95        10

    accuracy                           0.97        30
   macro avg       0.97      0.97      0.97        30
weighted avg       0.97      0.97      0.97        30

